# *Pipelines* en *Scikit-Learn*

## Pasos previos

In [ ]:
import pandas as pdimport numpy as npfrom sklearn.model_selection import train_test_splithousing = pd.read_csv("./data/housing.csv") train_set, test_set = train_test_split(housing, test_size=0.2,    stratify=pd.cut(housing["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5]),    random_state=42    )X_train = train_set.drop("median_house_value", axis=1)y_train = train_set["median_house_value"].copy()X_train_num = X_train.select_dtypes(include=[np.number]) # select numerical columns

## Pipelines de preprocesamiento

Un ***pipeline*** es una secuencia de componentes de procesamiento de datos. La clase `Pipeline` de scikit-learn nos permite crear objetos que representan estas secuencias, para poder aplicarlas posteriormente a cualquier conjunto de datos.

Todos los **estimadores** excepto el último deben ser **transformadores**. Cuando llamamos al método `fit` de la clase `Pipeline`, llama al método `fit_transform` de cada estimador secuencialmente, pasando la salida del método `transform` de un estimador al siguiente. El último estimador puede ser de cualquier tipo (transformador, clasificador, regresor, etc.).

Es importante tener claro qué significa cada [tipo de estimador en scikit-learn](./types_estimators.md).

Construyamos un *pipeline* que preprocese los predictores numéricos.

El constructor de la clase `Pipeline` recibe una lista de tuplas formadas por el nombre que identifica a cada estimador y ese estimador. Todos los estimadores deben ser **transformadores**, excepto el último, que puede ser cualquier tipo de estimador.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")), # imputar la mediana para los valores faltantes
    ("standardize", StandardScaler()), # estandarizar los valores
])
num_pipeline.steps

También podemos usar la función `make_pipeline`, que crea un *pipeline* como el anterior pero asignando automáticamente un nombre a cada estimador.

In [ ]:
from sklearn.pipeline import make_pipelinenum_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())num_pipeline.steps

[('simpleimputer', SimpleImputer(strategy='median')),
 ('standardscaler', StandardScaler())]

El método `fit()` del *pipeline* llama al método `fit_transform()` de cada transformador, pasando la salida de cada uno al siguiente, y finalmente llama al método `fit()` del último estimador.

El método `fit_transform()` del *pipeline* hace lo mismo, pero llama al método `fit_transform()` del último estimador.

In [ ]:
X_train_num_prepared = num_pipeline.fit_transform(X_train_num)X_train_num_prepared[:2].round(2)

array([[-0.94,  1.35,  0.03,  0.58,  0.64,  0.73,  0.56, -0.89],
       [ 1.17, -1.19, -1.72,  1.26,  0.78,  0.53,  0.72,  1.29]])

Para visualizar mejor lo que hace el *pipeline*, podemos reconstruir un dataframe con sus resultados.

In [ ]:
pd.DataFrame(X_train_num_prepared,
            columns=num_pipeline.get_feature_names_out(), # obtener los nombres de las columnas tras la transformación
            index=X_train_num.index).head(2)

In [ ]:
num_pipeline[1]

,copy,True
,with_mean,True
,with_std,True


In [ ]:
num_pipeline[:-1]

,steps,"[('simpleimputer', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False


In [ ]:
num_pipeline.named_steps["simpleimputer"]

,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False


Con el método `set_params` podemos cambiar el valor de un parámetro de un estimador.

In [ ]:
num_pipeline.set_params(simpleimputer__strategy="mean")

,steps,"[('simpleimputer', ...), ('standardscaler', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'mean'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


Para crear un *pipeline* que preprocese tanto los predictores numéricos como los categóricos, necesitamos un transformador que seleccione las columnas que queremos transformar. Scikit-learn proporciona la clase `ColumnTransformer` para esto.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

num_attribs = ["longitude", "latitude", "housing_median_age", "total_rooms",
               "total_bedrooms", "population", "households", "median_income"]
cat_attribs = ["ocean_proximity"]

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"), # imputar la moda para los valores faltantes
    OneHotEncoder(handle_unknown="ignore")) # codificar las categorías

preprocessing = ColumnTransformer([
    ("num", num_pipeline, num_attribs), # aplicar el pipeline numérico a las columnas numéricas
    ("cat", cat_pipeline, cat_attribs)], # aplicar el pipeline categórico a las columnas categóricas
    remainder="passthrough" # las columnas restantes se mantienen sin cambios)

El parámetro `remainder='passthrough'` indica que las columnas que no han sido seleccionadas para la transformación se pasarán directamente al *pipeline* final sin cambios. Si no se especifica, las columnas no seleccionadas serán eliminadas (por defecto, `remainder='drop'`). En este caso se están pasando todas, por lo que no habrá diferencia.

Para poder asignar Pipelines a todas las columnas según su tipo, podemos usar la función `make_column_transformer`.

In [ ]:
from sklearn.compose import make_column_selector, make_column_transformerpreprocessing = make_column_transformer(    (num_pipeline, make_column_selector(dtype_include=np.number)),    (cat_pipeline, make_column_selector(dtype_include=object)),)

y podemos usar el método `fit_transform()` del *pipeline* para transformar los datos de entrenamiento.

In [ ]:
housing_prepared = preprocessing.fit_transform(X_train)

In [ ]:
housing_prepared_fr = pd.DataFrame(    housing_prepared,    columns=preprocessing.get_feature_names_out(),    index=X_train.index)housing_prepared_fr.head(7).T

,12655,15502,2908,14053,20496,1481,18125
pipeline-1__longitude,-0.941350,1.171782,0.267581,1.221738,0.437431,-1.231094,-1.226099
pipeline-1__latitude,1.347438,-1.192440,-0.125972,-1.351474,-0.635818,1.085499,0.790817
pipeline-1__housing_median_age,0.027564,-1.722018,1.220460,-0.370069,-0.131489,-0.051963,-0.449595
pipeline-1__total_rooms,0.584777,1.261467,-0.469773,-0.348652,0.427179,-0.661977,0.747520
pipeline-1__total_bedrooms,0.638183,0.779415,-0.547672,-0.038752,0.270495,-0.688903,0.331371
pipeline-1__population,0.732602,0.533612,-0.674675,-0.467617,0.374060,-0.623583,0.324761
pipeline-1__households,0.556286,0.721318,-0.524407,-0.037297,0.220898,-0.652174,0.383269
pipeline-1__median_income,-0.893647,1.292168,-0.525434,-0.865929,0.325752,-0.094224,1.895358
pipeline-2__ocean_proximity_<1H OCEAN,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000
pipeline-2__ocean_proximity_INLAND,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000


In [ ]:
preprocessing.get_feature_names_out()

array(['pipeline-1__longitude', 'pipeline-1__latitude',
       'pipeline-1__housing_median_age', 'pipeline-1__total_rooms',
       'pipeline-1__total_bedrooms', 'pipeline-1__population',
       'pipeline-1__households', 'pipeline-1__median_income',
       'pipeline-2__ocean_proximity_<1H OCEAN',
       'pipeline-2__ocean_proximity_INLAND',
       'pipeline-2__ocean_proximity_ISLAND',
       'pipeline-2__ocean_proximity_NEAR BAY',
       'pipeline-2__ocean_proximity_NEAR OCEAN'], dtype=object)

## Próximos pasos

Este notebook presentó los conceptos fundamentales de los Pipelines de scikit-learn. El pipeline de preprocesamiento se desarrolla más en:

- [e2e051 - Transformadores Personalizados](e2e051_custom_transformers.ipynb): `FunctionTransformer` para ratios de características y transformaciones logarítmicas
- [e2e060 - Agrupamiento Espacial](e2e060_spatial_clustering.ipynb): transformador `ClusterSimilarity` para características geoespaciales

El pipeline de preprocesamiento completo se consolida en [`utils/housing_preprocessing.py`](utils/housing_preprocessing.py) para su reutilización en los notebooks de Entrenamiento del Modelo.